In [ ]:
from fouri import load_config

# This experiment's experiment.json (create or edit it with setup_experiment.ipynb)
CONFIG = r'Z:\PATH\TO\EXPERIMENT\experiment.json'

exp = load_config(CONFIG)

In [ ]:
import pickle
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn import preprocessing
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import pandas as pd
import phate
import math
import random
import gc
import scprep
from datetime import datetime, time
from matplotlib.animation import ImageMagickWriter
import matplotlib.animation as animation
import zipfile
from urllib.request import urlopen
import scipy.stats as st
from scipy.stats import norm
from scipy.stats import gaussian_kde
from scipy.stats import kde
from scipy.stats import binned_statistic
from scipy.stats import f_oneway
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
plt.rcParams['pdf.fonttype'] = 42
print(sns.__version__)
from anndata import AnnData
import scanpy as sc
from delve import *
import anndata as ad
from sklearn.preprocessing import MinMaxScaler
from sketchKH import sketch
from sklearn.cluster import KMeans
import umap
print(sc.__version__)
today = datetime.now().strftime("%m%d%Y-%H%M")

Filtering features from standard_adata_sub

In [ ]:
import anndata as ad

# Load existing AnnData
adata_path = exp.file('standard_adata_sub.h5ad')
adata = ad.read_h5ad(adata_path)

# Features you want to keep
# Same proteins as the original list, with round names from experiment 1.3's panel
# (total_pS6_S6_ratio, pAKT_AKT_ratio, p53 and ER aren't in this panel)
features_to_keep = [
    'total_area_DNA',
    'pRB_RB_ratio',
    'R1_CyclinA2_nuc_mean', 'R0_CDK2_nuc_mean', 'R3_CyclinE1_nuc_mean', 'R2_CyclinD1_nuc_mean',
    'R1_CDK4_nuc_mean', 'R3_p21_nuc_mean', 'R1_E2F1_nuc_mean', 'R2_CyclinB1_nuc_mean',
    'R4_CDT1_nuc_mean', 'R5_cMyc_nuc_mean',
]

# Check which requested features are present
present_features = [f for f in features_to_keep if f in adata.var_names]
missing_features = [f for f in features_to_keep if f not in adata.var_names]

print("Present features:", present_features)
print("Missing features:", missing_features)

# Subset to only the features that exist
adata_filtered = adata[:, present_features].copy()

print("Original shape:", adata.shape)
print("Filtered shape:", adata_filtered.shape)

# Save new filtered file
save_path = exp.file('standard_adata_sub_filtered.h5ad')
adata_filtered.write_h5ad(save_path)

In [ ]:
# Read back in the subsampled adata file
adata_save_path = exp.file('standard_adata_sub_filtered.h5ad')
standard_adata_sub = ad.read_h5ad(adata_save_path)

# 🔹 Filter only specific sample IDs right after loading


selected_sample_ids = [1, 2, 3, 4, 5]

standard_adata_sub = standard_adata_sub[
    standard_adata_sub.obs['sample_ID'].isin(selected_sample_ids)
].copy()


In [ ]:
standard_adata_sub.obs

In [ ]:
def laplacian_score_fs(adata = None,
                    k: int  = None,
                    n_jobs: int  = -1):

    X, feature_names, obs_names = parse_input(adata)
    W = construct_affinity(X = X, k = k, n_jobs = n_jobs)
    scores = laplacian_score(X = X, W = W)
    predicted_features = pd.DataFrame(scores, index = feature_names, columns = ['laplacian_score'])
    predicted_features = predicted_features.sort_values(by = 'laplacian_score', ascending = True)

    return predicted_features 

In [ ]:
#l_score_fullest = laplacian_score_fs(adata_fullest_sub, k = 200)
l_score_standard = laplacian_score_fs(standard_adata_sub, k = 100)
#l_score_normalized = laplacian_score_fs(normalized_trimmed_noPSTAT5_adata_sub, k = 200)

In [ ]:
len(l_score_standard)

In [ ]:
l_score_standard

Calculating PHATE Plot Coordinates

In [ ]:
import os
import numpy as np
import phate
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from pptx import Presentation
from pptx.util import Inches
from io import BytesIO
import pandas as pd

# Define parameter ranges
#count_list = [100]  # nearest neighbors
#index_ranges = [20]  # how many features it is considering
#t_values = ['auto']
#gamma_values = [1]

# Define parameter ranges
count_list = [80]  # nearest neighbors
index_ranges = [10]  # how many features it is considering
t_values = ['auto']
gamma_values = [1]

# Create a PowerPoint presentation object
presentation = Presentation()

# Loop over each combination of parameters
for count in count_list:
    for index_range in index_ranges:
        for t in t_values:
            for gamma in gamma_values:
                # Initialize PHATE operator with varying t and gamma, projecting to three dimensions
                phate_op = phate.PHATE(knn=count, t=t, gamma=gamma, n_components=3)

                # Fit-transform PHATE
                X_phate = phate_op.fit_transform(
                    standard_adata_sub.X[
                        :, np.isin(
                            standard_adata_sub.var_names,
                            l_score_standard.index[:index_range]
                        )
                    ]
                )

                # Set PHATE result for plotting
                standard_adata_sub.obsm['X_phate'] = X_phate

                # Generate plot title
                plot_title = f'PHATE: Neighbors={count}, Index Range={index_range}, t={t}, gamma={gamma}'

                # Define angles for rotation in degrees
                angles = [(30, 30), (30, 120), (30, 210), (30, 300)]

                # ----- Dose labels sorted numerically (handles strings like "10 nM") -----
                dose_series = standard_adata_sub.obs['treatment'].astype(str).str.strip()
                tmp = pd.DataFrame({'label': dose_series.unique()})
                tmp['num'] = tmp['label'].str.extract(r'([\d.]+)').astype(float)
                ordered_labels = tmp.sort_values('num', na_position='last')['label'].tolist()

                # Build colormap in numeric order
                cmap = plt.get_cmap('viridis', len(ordered_labels))
                colors = cmap(np.linspace(0, 1, len(ordered_labels)))
                label_to_color = dict(zip(ordered_labels, colors))
                # ------------------------------------------------------------------------

                # Create a new slide for the current set of plots
                slide = presentation.slides.add_slide(presentation.slide_layouts[5])
                title = slide.shapes.title
                title.text = plot_title

                for i, (elev, azim) in enumerate(angles, start=1):
                    fig = plt.figure()
                    ax = fig.add_subplot(111, projection='3d')

                    # Plot each label in numeric order with a unique color
                    handles = []
                    for label in ordered_labels:
                        color = label_to_color[label]
                        idx = (standard_adata_sub.obs['treatment'].astype(str).str.strip() == label)
                        h = ax.scatter(
                            X_phate[idx, 0], X_phate[idx, 1], X_phate[idx, 2],
                            color=color, label=label, s=5
                        )
                        handles.append(h)

                    ax.view_init(elev=elev, azim=azim)
                    ax.set_title(plot_title)
                    ax.legend(handles=handles, title='treatment', bbox_to_anchor=(1.05, 1), loc='upper left')

                    # Save the plot to a BytesIO object
                    img_stream = BytesIO()
                    plt.savefig(img_stream, format='png', bbox_inches='tight')
                    plt.close()
                    img_stream.seek(0)

                    # Add the plot to the current slide in a 2x2 grid
                    left = Inches(1 + ((i - 1) % 2) * 4)
                    top = Inches(1.5 + ((i - 1) // 2) * 3.5)
                    slide.shapes.add_picture(img_stream, left, top, width=Inches(3.5))

# Save the PowerPoint presentation
pptx_filename = exp.file('phate_plots_presentation_treat.pptx')
presentation.save(pptx_filename)

print(f"3D PHATE plots saved with multiple angles and legends. PowerPoint presentation saved as {pptx_filename}.")


For plotting different data onto 3D PHATE structures

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.colors as mcolors
from pptx import Presentation
from pptx.util import Inches, Cm
from io import BytesIO

# Define angles for rotation in degrees
angles = [(30, 30), (30, 120), (30, 210), (30, 300)]

# Ensure the directory for saving plots exists
plots_dir = exp.file('phate_plots')
os.makedirs(plots_dir, exist_ok=True)

# Create a PowerPoint presentation object
prs = Presentation()
slide_width = prs.slide_width
slide_height = prs.slide_height

# Get all features in .var_names
features = standard_adata_sub.var_names

# Loop over each feature
for feature in features:
    # Obtain expression data for the feature
    data = standard_adata_sub[:, feature].X

    # Cap high and low values for visualization
    percentile_99 = np.percentile(data, 99)
    percentile_1 = np.percentile(data, 1)
    data_capped = np.clip(data, percentile_1, percentile_99)

    # Normalize and map data to colors
    norm = plt.Normalize(np.min(data_capped), np.max(data_capped))
    cmap = plt.get_cmap('viridis')
    colors = cmap(norm(data_capped))

    # Create a single figure with a 2x2 grid of subplots
    fig = plt.figure(figsize=(16, 16))
    for i, (elev, azim) in enumerate(angles, start=1):
        ax = fig.add_subplot(2, 2, i, projection='3d')
        # Plot the feature data
        ax.scatter(standard_adata_sub.obsm['X_phate'][:, 0],
                   standard_adata_sub.obsm['X_phate'][:, 1],
                   standard_adata_sub.obsm['X_phate'][:, 2],
                   color=colors, label=feature, s=5)
        ax.view_init(elev=elev, azim=azim)
        ax.set_title(f'View Angle {i}: {feature}')

    # Save plot to a BytesIO object
    canvas = FigureCanvas(fig)
    canvas.draw()
    image_stream = BytesIO()
    plt.savefig(image_stream, format='png')
    plt.close(fig)

    # Add a slide to the presentation and remove default placeholders
    slide_layout = prs.slide_layouts[5]  # Using a blank slide layout
    slide = prs.slides.add_slide(slide_layout)
    for placeholder in slide.placeholders:
        placeholder.element.getparent().remove(placeholder.element)

    # Calculate image size to maximize slide use while keeping aspect ratio
    aspect_ratio = fig.get_figwidth() / fig.get_figheight()
    slide_aspect_ratio = slide_width / slide_height
    if aspect_ratio > slide_aspect_ratio:
        width = slide_width
        height = slide_width / aspect_ratio
    else:
        height = slide_height
        width = slide_height * aspect_ratio

    left = (slide_width - width) / 2
    top = (slide_height - height) / 2

    pic = slide.shapes.add_picture(image_stream, left, top, width=width, height=height)
    image_stream.close()

# Save the presentation
prs.save(f'{plots_dir}/phate_plots_presentation.pptx')
print("PowerPoint presentation saved with combined 3D PHATE plots optimized to fill slides.")


After you've decided on a final structure, you can save the ADATA file. It should have the PHATE coordinates saved as part of it now, removing the need to recompute the phate structure if you want to plot different things onto it later

In [ ]:
#Save the entire adata file with new PHATE embeddings
adata_save_path = exp.file('standard_adata_sub_filtered_PHATE_80_10.h5ad')
standard_adata_sub.write_h5ad(adata_save_path)

In [ ]:
import anndata as ad

file_path = exp.file('standard_adata_sub_filtered_PHATE_80_10.h5ad')

adata = ad.read_h5ad(file_path, backed='r')

print(adata.var_names)
print(adata.obs.columns.tolist())
